In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: Isang Qiskit Function ng Global Data Quantum


*Tingnan ang [API reference](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Ang mga Qiskit Function ay isang eksperimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ang mga ito at maaaring magbago.
## Pangkalahatang-ideya
Ang Quantum Portfolio Optimizer ay isang Qiskit Function na tumutugon sa dynamic portfolio optimization na problema — isang karaniwang problema sa pananalapi na naglalayong muling i-balance ang mga pana-panahong pamumuhunan sa iba't ibang asset upang mapakinabangan ang kita at mabawasan ang panganib. Sa pamamagitan ng paggamit ng pinakabagong mga teknik sa quantum optimization, pinapasimple ng function na ito ang proseso para makinabang ang mga gumagamit — kahit walang kaalaman sa quantum computing — mula sa mga kalamangan nito sa paghanap ng pinakamainam na mga trajectory ng pamumuhunan. Perpekto para sa mga portfolio manager, mananaliksik sa quantitative finance, at mga indibidwal na mamumuhunan, pinapahintulutan ng tool na ito ang back-testing ng mga trading strategy sa portfolio optimization.
## Paglalarawan ng function
Ginagamit ng Quantum Portfolio Optimizer function ang Variational Quantum Eigensolver (VQE) algorithm para malutas ng Quadratic Unconstrained Binary Optimization (QUBO) na problema, na tumutugon sa mga dynamic portfolio optimization na problema. Kailangan lamang ng mga gumagamit na ibigay ang datos ng presyo ng asset at tukuyin ang constraint ng pamumuhunan, pagkatapos ay pinapatakbo ng function ang proseso ng quantum optimization na nagbabalik ng isang set ng mga na-optimize na trajectory ng pamumuhunan.

Ang proseso ay binubuo ng apat na pangunahing yugto. Una, ang input data ay imi-map sa isang quantum-compatible na problema, itinatayo ang QUBO ng dynamic portfolio optimization na problema, at binabago ito sa isang quantum operator (Ising Hamiltonian). Susunod, ang input na problema at ang VQE algorithm ay ina-adapt para patakbuhin sa quantum hardware. Pagkatapos ay pinapatakbo ang VQE algorithm sa quantum hardware, at sa wakas, ang mga resulta ay pinoproseso upang maibigay ang pinakamainam na mga trajectory ng pamumuhunan. Kasama rin sa sistema ang isang noise-aware ([SQD](/guides/qiskit-addons-sqd)-based) na post-processing upang mapakinabangan ang kalidad ng output.

Ang Qiskit Function na ito ay batay sa [nailathala na manuskrito](https://arxiv.org/abs/2412.19150) ng Global Data Quantum.
![Visualization ng workflow ng function](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Magsimula
Mag-authenticate gamit ang iyong [API key](https://quantum.cloud.ibm.com/) at piliin ang Qiskit Function tulad ng sumusunod. (Ipinapalagay ng snippet na ito na nai-save mo na ang iyong [account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na kapaligiran.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## Halimbawa: Dynamic portfolio optimization na may pitong asset
Ipinapakita ng halimbawang ito kung paano isagawa ang dynamic portfolio optimization (DPO) function at i-adjust ang mga setting nito para sa pinakamainam na pagganap. Kasama nito ang mga detalyadong hakbang para i-fine-tune ang mga parameter upang makamit ang mga nais na resulta.

Ang kasong ito ay kinabibilangan ng pitong asset, apat na time step, at apat na resolution qubit, na nagresulta sa kabuuang pangangailangan na 112 qubit.
### 1. Basahin ang mga asset na kasama sa portfolio.
Kung ang lahat ng asset sa portfolio ay nakaimbak sa isang folder sa isang partikular na path, maaari mo itong i-load sa isang `pandas.DataFrame` at i-convert ito sa isang object na may format na `dict` gamit ang sumusunod na function.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

Para sa halimbawang ito, ginamit namin ang mga asset na [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y), at [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Inilalarawan ng sumusunod na figure ang datos na ginamit sa halimbawang ito, na nagpapakita ng ebolusyon ng pang-araw-araw na closing price ng mga asset mula 1 Enero hanggang 1 Setyembre 2023.

Sa halimbawang ito, para matiyak ang pagkakatugma sa iba't ibang petsa, pinuno namin ang mga araw na hindi nagtatrabaho ng closing price mula sa nakaraang available na petsa. Inilalapat namin ang hakbang na ito dahil ang mga napiling asset ay nagmumula sa iba't ibang merkado na may magkakaibang araw ng pangangalakal, na ginagawa itong mahalaga na i-standardize ang dataset para sa konsistensya.
![Visualization ng makasaysayang datos ng mga asset](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Tukuyin ang problema.
Tukuyin ang mga detalye ng problema sa pamamagitan ng pag-configure ng mga parameter sa dictionary na `qubo_settings`.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. Tukuyin ang mga setting ng optimizer at ansatz (Opsyonal)
Opsyonal na tukuyin ang mga tiyak na pangangailangan para sa proseso ng optimization, kasama ang pagpili ng optimizer at mga parameter nito, pati na rin ang pagtukoy ng primitive at mga konfigurasyong nito.

Para sa Tailored Ansatz, ang napiling laki ng populasyon ay batay sa mga nakaraang eksperimento na nagpapakita na ang halagang ito ay nagbubunga ng stable at mahusay na optimization.

Sa kaso ng Real Amplitudes Ansatz, maaari kang sumunod sa linear na relasyon sa pagitan ng ``population_size`` at ng bilang ng mga qubit sa circuit. Bilang isang tinatayang rule of thumb, inirerekomenda na gumamit ng **minimum** na ``population_size ~ 0.8 * n_qubits`` para sa `real_amplitudes` ansatz.

Inaasahang magkakaroon ang Optimized Real Amplitudes ng mas mahusay na optimization performance kaysa sa Real Amplitudes ansatz. Gayunpaman, ang bilang ng mga variable na ino-optimize sa ansatz na ito ay tumataas nang mas mabilis kaysa sa kaso ng Real Amplitudes (tingnan ang [manuskrito](https://arxiv.org/pdf/2412.19150)). Kaya, para sa malalaking problema, ang Optimized Real Amplitudes ay nangangailangan ng mas maraming circuit execution. Ang Optimized Real Amplitudes ay malamang na kapaki-pakinabang para sa mga problemang nangangailangan ng hanggang 100 qubit, ngunit inirerekomenda na maging maingat kapag nagtatakda ng mga parameter na ``population_size``. Bilang halimbawa ng scale-up na ito sa ``population_size``, ang nakaraang talahanayan ay nagpapakita na para sa isang 84-qubit na problema, ang Optimize Real Amplitudes ay nangangailangan ng 120 na ``population_size``, habang para sa isang 56-qubit na problema, sapat na ang ``population_size`` na 40.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

Posible rin na pumili ng isang tiyak na ansatz. Gumagamit ang sumusunod ng ansatz na ``'Tailored'``.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. Patakbuhin ang problema.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Kunin ang mga resulta
Ang function ay nagbabalik ng dictionary na may mga investment trajectory na nakaayos mula sa pinakamababa hanggang pinakamataas ayon sa halaga ng kanilang objective function (tingnan ang seksyon ng [Output](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) sa API reference). Ang set ng mga resulta na ito ay nagbibigay-daan sa pagkilala ng trajectory na may pinakamababang gastos at ang kaukulang mga pagsusuri ng pamumuhunan. Bukod pa rito, nagbibigay ito para sa pagsusuri ng iba't ibang trajectory, na nagpapadali sa pagpili ng mga pinaka-angkop sa mga tiyak na pangangailangan o layunin. Tinitiyak ng flexibility na ito na ang mga pagpipilian ay maaaring iayon upang tumugon sa iba't ibang kagustuhan o sitwasyon.
Simulan sa pamamagitan ng pagpapakita ng resultang estratehiya na nakamit ang pinakamababang objective cost na natagpuan sa proseso.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


Pagkatapos, gamit ang metadata, maaari kang ma-access ang mga resulta ng lahat ng mga na-sample na estratehiya. Maaari mong pag-aralan pa ang mga alternatibong trajectory na ibinalik ng optimizer. Para gawin ito, basahin ang dictionary na nakaimbak sa `dpo_result['metadata']['all_samples_metrics']`, na naglalaman hindi lamang ng karagdagang impormasyon tungkol sa pinakamainam na estratehiya, kundi pati na rin ang mga detalye ng iba pang mga kandidatong estratehiya na sinuri sa panahon ng optimization.

Ang sumusunod na halimbawa ay nagpapakita kung paano basahin ang impormasyong ito gamit ang `pandas` upang kunin ang mga pangunahing sukatan na nauugnay sa pinakamainam na estratehiya. Kasama dito ang Restriction Deviation, Sharpe Ratio, at ang kaukulang investment return.